In [1]:
import os
os.chdir('/home/smallyan/eval_agent')
print(f"Working directory: {os.getcwd()}")

Working directory: /home/smallyan/eval_agent


# Replicator-Documentation Evaluator

This notebook evaluates whether the replicator's documentation faithfully reproduces the results and conclusions of the original experiment.

## Task Overview
1. Read original documentation from `/net/scratch2/smallyan/arithmetic_eval/`
2. Read replicated documentation from `/net/scratch2/smallyan/arithmetic_eval/evaluation/replications/`
3. Evaluate DE1-DE3 criteria
4. Generate evaluation summary files

In [2]:
# Check GPU availability
import torch
if torch.cuda.is_available():
    print(f"GPU available: {torch.cuda.get_device_name(0)}")
    print(f"CUDA version: {torch.version.cuda}")
else:
    print("No GPU available, using CPU")

GPU available: NVIDIA A100 80GB PCIe
CUDA version: 12.4


In [3]:
# Define paths
ORIGINAL_REPO = "/net/scratch2/smallyan/arithmetic_eval"
REPLICATION_DIR = "/net/scratch2/smallyan/arithmetic_eval/evaluation/replications"
OUTPUT_DIR = "/net/scratch2/smallyan/arithmetic_eval/evaluation/replication_eval"

# List contents to understand the structure
import os

print("=== Original repo contents ===")
if os.path.exists(ORIGINAL_REPO):
    for item in os.listdir(ORIGINAL_REPO):
        print(f"  {item}")
else:
    print(f"Path does not exist: {ORIGINAL_REPO}")

print("\n=== Replication directory contents ===")
if os.path.exists(REPLICATION_DIR):
    for item in os.listdir(REPLICATION_DIR):
        print(f"  {item}")
else:
    print(f"Path does not exist: {REPLICATION_DIR}")

=== Original repo contents ===
  evaluation
  documentation.pdf
  CodeWalkthrough.md
  .git
  plan.md
  .gitignore
  data
  cache
  LICENSE
  figures
  scripts
  arithmetic_claude_2025-12-23_02-35-29

=== Replication directory contents ===
  accuracy_comparison.png
  replication.ipynb
  documentation_replication.md
  evaluation_replication.md
  self_replication_evaluation.json


In [4]:
# Check for original documentation (PDF and markdown files)
print("Looking for original documentation files...")

# Check documentation.pdf
doc_pdf_path = os.path.join(ORIGINAL_REPO, "documentation.pdf")
if os.path.exists(doc_pdf_path):
    print(f"Found: {doc_pdf_path}")
    
# Check for any markdown documentation
for fname in os.listdir(ORIGINAL_REPO):
    if fname.endswith('.md'):
        fpath = os.path.join(ORIGINAL_REPO, fname)
        print(f"Found markdown: {fpath}")
        
# Also check CodeWalkthrough.md
walkthrough_path = os.path.join(ORIGINAL_REPO, "CodeWalkthrough.md")
if os.path.exists(walkthrough_path):
    print(f"\nCodeWalkthrough.md exists")

Looking for original documentation files...
Found: /net/scratch2/smallyan/arithmetic_eval/documentation.pdf
Found markdown: /net/scratch2/smallyan/arithmetic_eval/CodeWalkthrough.md
Found markdown: /net/scratch2/smallyan/arithmetic_eval/plan.md

CodeWalkthrough.md exists


In [5]:
# Read the original documentation (CodeWalkthrough.md serves as documentation)
with open(os.path.join(ORIGINAL_REPO, "CodeWalkthrough.md"), 'r') as f:
    original_walkthrough = f.read()

print("=== Original CodeWalkthrough.md ===")
print(original_walkthrough[:3000])
print("\n... [truncated]" if len(original_walkthrough) > 3000 else "")

=== Original CodeWalkthrough.md ===
# Vector Arithmetic in Concept and Token Subspaces
*Code and data for short paper at the NeurIPS 2025 Mechanistic Interpretability Workshop. See paper website [here](https://arithmetic.baulab.info).*

In this work, we use the weights of concept and token induction heads discovered in ["The Dual-Route Model of Induction"](https://dualroute.baulab.info/) to analyze word embeddings. We find that using these heads to "focus" on semantic information can make word2vec-style analogies like `Athens - Greece + China = Beijing` work out much more cleanly than they do using raw hidden states. Doing the same with token induction heads can help with more wordform-focused word2vec tasks, like `dance - dancing + coding = code`. 

# Data
We use two datasets in this work, which each have a number of tasks. 
1. `word2vec` - original data from [Mikolov et al. (2013)](https://arxiv.org/pdf/1301.3781)
2. `fvs` - function vector tasks from [Todd et al. (2024)](https://fun

In [6]:
# Read the plan.md for additional context about the original project
with open(os.path.join(ORIGINAL_REPO, "plan.md"), 'r') as f:
    original_plan = f.read()

print("=== Original plan.md ===")
print(original_plan)

=== Original plan.md ===
# Plan
## Objective
Show that concept and token induction heads can identify subspaces of Llama-2-7b activations with coherent semantic and surface-level structure, enabling more accurate parallelogram arithmetic (e.g., Athens – Greece + China = Beijing) than using raw hidden states.

## Hypothesis
1. Poor parallelogram arithmetic results on raw Llama-2-7b hidden states are due to interference from irrelevant information in model activations.
2. Word2vec arithmetic is only effective when performed in a semantic subspace of model activations, not on the full hidden state space.
3. Concept and token induction heads operate in subspaces that represent different facets of words (semantic vs. surface-level).

## Methodology
1. Build concept and token lenses by summing OV matrices (O(l,h)V(l,h)) from top-k concept/token induction heads identified in prior work, creating transformations LCk and LTk.
2. Extract word embeddings by passing single words (optionally with t

In [7]:
# Read the replicated documentation
with open(os.path.join(REPLICATION_DIR, "documentation_replication.md"), 'r') as f:
    replicated_doc = f.read()

print("=== Replicated documentation_replication.md ===")
print(replicated_doc)

=== Replicated documentation_replication.md ===
# Documentation: Replication of Vector Arithmetic in Concept and Token Subspaces

## Goal

Replicate the experiment demonstrating that concept and token induction heads can identify subspaces of Llama-2-7b activations with coherent semantic and surface-level structure, enabling more accurate parallelogram arithmetic (e.g., Athens – Greece + China = Beijing) than using raw hidden states.

## Data

### Datasets Used
1. **word2vec dataset** (from Mikolov et al., 2013):
   - `capital-common-countries`: 506 capital-country pairs
   - `family`: 506 family relation pairs  
   - `gram5-present-participle`: Present participle transformations
   - `gram7-past-tense`: Past tense transformations
   - Plus 10 additional tasks covering semantic and grammatical categories

2. **Head Ordering Scores**:
   - `concept_copying_len30_n1024.json`: Causal scores for concept heads
   - `token_copying_len30_n1024.json`: Causal scores for token heads
   - These a

In [8]:
# Also check the evaluation_replication.md for more context
with open(os.path.join(REPLICATION_DIR, "evaluation_replication.md"), 'r') as f:
    eval_replication = f.read()

print("=== Evaluation Replication MD ===")
print(eval_replication)

=== Evaluation Replication MD ===
# Evaluation: Replication of Vector Arithmetic in Concept and Token Subspaces

## Reflection

This replication attempt evaluated the reproducibility of the "Vector Arithmetic in Concept and Token Subspaces" experiment, which demonstrates that projecting Llama-2-7b hidden states through concept or token induction head OV matrices improves word2vec-style parallelogram arithmetic.

### What Worked Well
1. **Clear plan and code walkthrough**: The repository provided a well-structured plan.md explaining the hypothesis, methodology, and expected results
2. **Self-contained codebase**: All necessary scripts (parallelograms.py, all_parallelograms.py) and data files were present
3. **Pre-computed head orderings**: The causal scores for concept/token heads were cached, avoiding the need to recompute expensive head selection
4. **Cached results for verification**: Existing cached results allowed verification of reimplemented code

### Challenges Encountered
1. **